# 📝 과제 02: 온라인 쇼핑 상담 API 보호하기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행:</strong> RunPod 파드 하나에서 vLLM 서버(터미널 1), 과제 FastAPI 서버(터미널 2), 이 노트북을 모두 실행합니다. <strong style="font-size:1em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">작성:</strong> 서버 코드는 이 과제 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code> 패키지 .py 파일에 씁니다. 노트북은 실행 중인 서버(127.0.0.1:8080)에 요청을 보내 자가채점만 합니다.<br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 교안 02를 마친 상태에서 시작합니다. 이 과제 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api</code>에는 과제 01의 완성 코드가 들어 있으므로, 과제 01을 끝내지 못했어도 진행할 수 있습니다.</div>

## 만들 결과

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02의 보호 계층을 과제 01의 온라인 쇼핑 상담 API에 적용합니다. 상담 주소에는 서비스 키 인증과 요금제별 요청 한도를 붙이고, 모든 오류를 한 양식으로 돌려줍니다. 끝으로 코드는 고치지 않고 환경 변수만 바꿔, vLLM에 닿지 않을 때(502)와 늦을 때(504)를 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/secure_main.py</code>: 키 없이 열리는 상태 확인 주소와, 요청 한도 의존성을 붙인 쇼핑 상담 주소</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/dependencies/security.py</code>: 서비스 키 확인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_service_key</code>(401)와 요청 한도 확인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>(429와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code>)</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/core/errors.py</code>: 오류 양식 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;error&quot;: {&quot;code&quot;: ..., &quot;message&quot;: ...}}</code>의 처리 함수 다섯 개와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">register_error_handlers</code></li>
<li style="margin:3px 0;">RunPod 터미널 2 명령 두 개: 틀린 vLLM 주소로 실행(502), 시간 제한 1초로 실행(504)</li>
</ul>
</div>

## 준비: 서버와 제공 코드

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code>의 폴더 구조는 과제 01과 같고(교안 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code>과도 같습니다), 이 과제에서 아래 파일을 더 씁니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">폴더, 파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 과제에서</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">앱 조립: 과제 01의 주소 묶음에 보호 계층을 붙인 앱</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1번 작성</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/security.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서비스 키 확인과 요청 한도</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2번 작성</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/errors.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">오류 응답 양식</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3번 작성</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/service_keys.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">연습용 서비스 키</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">제공</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경:</strong> VS Code Remote SSH로 연 RunPod 원격 창의 노트북과 터미널을 씁니다. 터미널 1의 vLLM 서버는 교안 02에서 쓰던 것을 그대로 둡니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 연습용 서비스 키는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/resources/service_keys.json</code>에 있습니다. free 요금제 키 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-free-demo-0001</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-free-demo-0003</code>은 60초에 3번, pro 요금제 키 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-pro-demo-0002</code>는 60초에 30번까지 요청할 수 있습니다. 파일 모양은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;plans&quot;: {요금제: {&quot;requests_per_minute&quot;: n}}, &quot;keys&quot;: {키: {&quot;user&quot;: ..., &quot;plan&quot;: ...}}}</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong><ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/security.py</code>의 헤더 읽기 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key_header</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code>), 키 목록 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_KEYS</code>, 시간 창 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">WINDOW_SECONDS</code>(60초), 요청 기록 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_log</code></li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/errors.py</code>의 오류 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ERROR_CODES</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">handle_vllm_status_error</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code>의 앱 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">register_error_handlers(app)</code> 호출</li>
<li style="margin:3px 0;">과제 01의 완성 코드(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/chat.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/shop_router.py</code>)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router</code>도 이 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api</code>에 들어 있어 그대로 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/security.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/errors.py</code>에만 있습니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 02와 다른 점:</strong> 주소는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/shop</code>이고 서비스 키 이름이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-</code>으로 시작합니다. 키 확인, 요청 한도, 오류 양식의 규칙은 교안 02와 같습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong><ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code>의 .py 파일에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code> 주석 아래를 채웁니다. 요구사항 (n)은 그 파일의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code>에 씁니다.</li>
<li style="margin:3px 0;">저장한 뒤 그 문항의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[자가채점]</code> 셀을 실행합니다. 1번부터 차례로 풉니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 기록:</strong> 요청 한도는 서버 메모리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_log</code>로 세므로, 서버가 다시 시작하면 기록이 비워집니다. .py를 저장해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>가 다시 띄울 때도 같습니다. 자가채점은 일반 요청에 pro 키를 쓰고, free 키는 2번과 3번의 한도 확인에만 씁니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-free-demo-0003</code>은 2번에서 한 번만).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버가 응답하지 않을 때:</strong> 자가채점 셀이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ConnectError</code>나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ReadTimeout</code>으로 멈추면 터미널 2의 서버가 떠 있지 않거나 코드를 읽다가 오류가 난 것입니다. 터미널 2의 마지막 오류 줄에서 파일 이름과 줄 번호를 확인합니다.</li>
</ul>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 주소 ----
import json
from pathlib import Path

import httpx

lesson_dir = Path(".").resolve()  # 이 노트북의 단원 폴더(정답 노트북은 "..")
day_dir = lesson_dir.parent                # data, images 가 있는 일차 폴더
data_dir = day_dir / "data"

SERVING_URL = "http://127.0.0.1:8080"   # 터미널 2에서 띄운 FastAPI 서빙 API

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(서빙 벤치마킹 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# 자가채점에 쓸 쇼핑 질문(첫 질문 하나)
commerce_questions = load_jsonl(data_dir / "과제" / "commerce_questions.jsonl")
sample_question = commerce_questions[0]["question"]

# 과제 02 의 연습용 서비스 키(서버 패키지의 resources/service_keys.json 과 같습니다)
FREE_KEY = "shop-free-demo-0001"    # free 요금제: 60초에 3번
PRO_KEY = "shop-pro-demo-0002"      # pro 요금제: 60초에 30번
FREE_KEY_2 = "shop-free-demo-0003"  # 다른 free 키: 한도를 키마다 따로 세는지 확인할 때만 씁니다

# 상담 주소와 서버 응답을 기다리는 최대 시간(초, 서버의 vLLM 시간 제한 120초보다 길게)
CHAT_URL = f"{SERVING_URL}/v1/shop/chat"
STREAM_URL = f"{SERVING_URL}/v1/shop/chat/stream"
REQUEST_TIMEOUT = 180
print("자가채점 질문:", sample_question)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버(터미널 1):</strong> 교안 02의 vLLM 서버가 떠 있으면 그대로 씁니다. 서버가 없을 때만(새 파드에서 시작할 때 등) RunPod 터미널 1에서 아래 두 블록을 차례로 실행합니다. 첫 블록은 교안 01에서 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 vLLM 키를 불러옵니다(새 터미널에는 키가 없습니다).</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API"
pwd   # 끝이 day77_FastAPI_모델서빙API이면 맞습니다
mkdir -p output   # 서버 로그를 남길 폴더입니다
# 원격 저장소의 .env에 저장해 둔 vLLM 키를 불러옵니다(키를 새로 생성하지 않고, 화면에 찍지 않습니다)
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 vLLM 서버가 요청을 받을 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 둡니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 몇 분이 걸릴 수 있습니다.</p>
<p style="margin:6px 0;">서버 로그 첫머리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">non-default args</code> 줄에 vLLM 키가 찍히므로, 서버 로그나 터미널 1 화면은 다른 사람에게 공유하지 않습니다.</p>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">과제 서버(터미널 2):</strong> 터미널 2에 떠 있는 서버(교안 02의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code> 또는 과제 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.main</code>)를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끕니다. 아래 두 블록으로 이 과제 폴더로 이동한 뒤 같은 포트 8080에 과제 02 앱을 띄웁니다. TODO를 채우기 전에도 서버는 시작합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API/과제_02_온라인쇼핑_상담_API_보호하기"
pwd   # 끝이 과제_02_온라인쇼핑_상담_API_보호하기이면 맞습니다
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn shop_api.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
</div>

## 1. 키 없이 여는 주소와 보호하는 주소

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 주소 묶음을 앱에 붙일 때 묶음마다 의존성을 다르게 줍니다. 상태 확인 주소는 자동 점검이 키 없이 부르고, 상담 주소는 서비스 키를 받은 고객만 쓰게 합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/secure_main.py</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">system_router</code> 붙이기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">include_router</code>로 의존성 없이 붙입니다. 키 없이도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /healthz</code>가 200과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;status&quot;: &quot;ok&quot;}</code>를 돌려주고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>의 이 주소에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">security</code> 항목이 없습니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">shop_router</code> 붙이기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">include_router</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Depends(check_rate_limit)</code> 하나를 담은 리스트를 줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>이 키 확인을 먼저 거치므로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat/stream</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">security</code> 항목 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[{&quot;APIKeyHeader&quot;: []}]</code>가 붙습니다.</li>
</ul>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 02의 2절(서비스 키 인증)을 참고합니다.
세부구현
1) 앱의 include_router 메서드를 묶음마다 한 번씩 부릅니다.
2) 보호할 묶음에만 dependencies 키워드 인자로 의존성 리스트를 줍니다. 의존성은 Depends로 감쌉니다.
```

</details>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# [자가채점] 앱 제목, 키 없이 열리는 상태 확인 주소, 상담 주소의 보안 항목을 확인합니다
openapi = httpx.get(f"{SERVING_URL}/openapi.json", timeout=REQUEST_TIMEOUT).json()
assert openapi["info"]["title"] == "온라인 쇼핑 상담 API(서비스 키 필요)", f"앱 제목이 {openapi['info']['title']!r}입니다. 터미널 2에서 이 과제 폴더의 과제 02 앱(secure_main.py 의 app)을 실행했는지 확인하세요."

# (1) 상태 확인 주소: 키 없이 200, 보안 항목 없음
health = httpx.get(f"{SERVING_URL}/healthz", timeout=REQUEST_TIMEOUT)
assert health.status_code == 200 and health.json() == {"status": "ok"}, f"키 없는 GET /healthz 가 {health.status_code}입니다. secure_main.py 의 TODO (1)에서 system_router 를 의존성 없이 붙이세요."
assert "security" not in openapi["paths"]["/healthz"]["get"], "GET /healthz 에 보안 항목이 붙었습니다. system_router 에는 dependencies 를 주지 마세요(TODO (1))."

# (2) 상담 주소 두 개: 서비스 키 보안 항목
for path in ["/v1/shop/chat", "/v1/shop/chat/stream"]:
    assert path in openapi["paths"], f"주소 목록에 {path} 가 없습니다. secure_main.py 의 TODO (2)에서 shop_router 를 붙였는지 확인하세요. 이 폴더의 routers/shop_router.py 는 과제 01 완성 코드이므로 고치지 않습니다."
    security = openapi["paths"][path]["post"].get("security")
    assert security == [{"APIKeyHeader": []}], f"{path} 의 보안 항목이 {security}입니다. include_router 의 dependencies 에 Depends(check_rate_limit) 를 담으세요(TODO (2))."
print("✅ 통과!")

## 2. 서비스 키와 요청 한도

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 키가 없는 요청과 요금제 한도를 넘긴 요청을 vLLM에 보내기 전에 막아, GPU를 정해진 고객이 정해진 만큼만 쓰게 합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/dependencies/security.py</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">require_service_key(api_key)</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더의 키가 없거나(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_KEYS[&quot;keys&quot;]</code>에 없으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPException</code>을 상태 코드 401로 일으킵니다. 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;key&quot;: 키, &quot;user&quot;: 사용자, &quot;plan&quot;: 요금제}</code> 딕셔너리를 돌려줍니다. 의존성은 본문 검사보다 먼저 실행되므로, 키가 없으면 본문이 틀린 요청도 401입니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">check_rate_limit(client)</code>:</strong> 그 키가 60초 창 안에 보낸 요청 수로 한도를 확인합니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_log</code>에 든 그 키의 요청 시각(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.monotonic</code> 값) 가운데 지금과의 차이가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">WINDOW_SECONDS</code>(60초)보다 작은 것만 남깁니다.</li>
<li style="margin:3px 0;">남은 수가 요금제의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests_per_minute</code> 이상이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPException</code>을 상태 코드 429로 일으킵니다. 헤더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code>에는 가장 오래된 요청이 60초 창을 벗어날 때까지 남은 초를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">math.ceil</code>로 올려 정수 글자로 담습니다.</li>
<li style="margin:3px 0;">통과한 요청만 기록하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">client</code>를 돌려줍니다. 그래서 free 키는 60초에 세 번까지 200, 네 번째는 429이고, pro 키는 네 번째도 통과합니다.</li>
<li style="margin:3px 0;">429를 받은 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code> 초만큼 기다려 다시 보내면 200입니다. 요청 수는 키마다 따로 세므로, 한 free 키가 한도를 다 써도 다른 free 키는 첫 요청부터 200입니다.</li>
<li style="margin:3px 0;">자가채점은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code> 초만큼 기다리므로 1분 남짓 멈춰 있다가 끝납니다.</li>
</ul>
</li>
</ul>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 02의 2절(서비스 키 인증)과 3절(요청 한도)을 참고합니다.
세부구현
1) 키 확인은 if 문 하나로 None인 경우와 키 목록에 없는 경우를 함께 봅니다.
2) 한도 확인은 for 문으로 창 안의 시각만 새 리스트에 담고, len으로 센 수를 한도와 비교합니다.
3) 거절할 때는 HTTPException의 headers 키워드 인자에 Retry-After를 글자로 담습니다.
4) 통과한 요청은 지금 시각을 리스트에 더해 request_log의 그 키 자리에 다시 저장합니다.
```

</details>

In [ ]:
# [자가채점] 키가 없거나 틀린 요청(401), free 키의 한도(429와 Retry-After), 기다린 뒤 다시 받기, 키별 한도, pro 키의 한도를 확인합니다
import math
import time

no_key = httpx.post(CHAT_URL, json={"question": sample_question}, timeout=REQUEST_TIMEOUT)
assert no_key.status_code == 401, f"키 없는 요청이 {no_key.status_code}입니다. 404 면 secure_main.py 의 TODO (2)에서 shop_router 를 붙였는지, 200 이면 secure_main.py 의 TODO (2)의 dependencies 를, 500 이면 dependencies/security.py 의 TODO (1)을 확인하세요."
wrong_key = httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": "shop-wrong-key"}, timeout=REQUEST_TIMEOUT)
assert wrong_key.status_code == 401, f"목록에 없는 키의 요청이 {wrong_key.status_code}입니다. 키가 SERVICE_KEYS[\"keys\"] 에 있는지도 확인하세요(TODO (1))."
no_key_bad_body = httpx.post(CHAT_URL, json={"question": ""}, timeout=REQUEST_TIMEOUT)
assert no_key_bad_body.status_code == 401, f"키도 없고 본문도 틀린 요청이 {no_key_bad_body.status_code}입니다. 키 확인은 의존성으로 붙여 본문 검사보다 먼저 실행되게 하세요."

# free 키: 같은 질문 네 번. 60초에 3번이므로 200, 200, 200, 429(요청마다 보낸 시각을 함께 남깁니다)
free_responses = []
sent_at = []
for count in range(4):
    sent_at.append(time.monotonic())
    free_responses.append(httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": FREE_KEY}, timeout=REQUEST_TIMEOUT))

    # 첫 세 요청 사이를 4초씩 벌립니다. 가장 오래된 요청이 아닌 다른 시각으로 계산한 Retry-After 는 8초 넘게 어긋나 아래 검사에 걸립니다
    if count < 2:
        time.sleep(4)
free_statuses = []
for response in free_responses:
    free_statuses.append(response.status_code)
assert free_statuses[0] != 429, "free 키를 60초 안에 이미 세 번 썼습니다. 터미널 2의 서버를 다시 시작하거나(Ctrl+C 뒤 같은 명령) 60초 뒤에 이 셀을 다시 실행하세요."
assert free_statuses == [200, 200, 200, 429], f"free 키 요청 네 번의 상태 코드가 {free_statuses}입니다. 60초 창 안의 요청 수가 한도(3) 이상일 때만 429 로 거절하고, 통과한 요청만 기록하세요(TODO (2))."

# 429 응답의 Retry-After: 1 이상 60 이하의 정수 글자
retry_after = free_responses[3].headers.get("retry-after")
assert retry_after is not None, "429 응답에 Retry-After 헤더가 없습니다. HTTPException 의 headers 에 Retry-After 를 담으세요(TODO (2)). core/errors.py 를 이미 채웠다면 handle_http_error 가 exc.headers 를 넘기는지도 확인하세요."
assert retry_after.isdigit() and 1 <= int(retry_after) <= 60, f"Retry-After 가 {retry_after!r}입니다. 가장 오래된 요청이 60초 창을 벗어날 때까지 남은 초를 math.ceil 로 올린 정수를 글자로 담으세요."

# Retry-After 값: 첫 요청부터 네 번째 요청까지 지난 시간을 60초에서 뺀 값과 3초 안으로 같아야 합니다
expected_retry = math.ceil(60 - (sent_at[3] - sent_at[0]))
assert abs(int(retry_after) - expected_retry) <= 3, f"Retry-After 가 {retry_after}초인데, 가장 오래된 요청 기준으로 계산하면 약 {expected_retry}초입니다. 창 안에 남은 요청 가운데 가장 오래된 요청(리스트의 첫 값)으로 남은 초를 계산하세요(dependencies/security.py 의 TODO (2))."

# 다른 free 키: 요청 수는 키마다 따로 세므로, 첫 키가 한도에 닿은 지금도 첫 요청은 200 입니다
other_free = httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": FREE_KEY_2}, timeout=REQUEST_TIMEOUT)
assert other_free.status_code == 200, f"{FREE_KEY} 가 한도에 닿은 직후, 처음 쓰는 다른 free 키 {FREE_KEY_2} 의 요청이 {other_free.status_code}입니다. 429 면 요청 수를 키마다 따로 세지 않고 모든 키를 한 기록으로 함께 세고 있습니다. request_log 에 키마다 따로 기록하고 그 키의 기록만 세세요(TODO (2))."

# 5초 뒤 첫 free 키로 세 번 더 보냅니다. 창 안에 통과한 요청이 3개라 모두 429 입니다
# (받아들인 요청보다 늦게 보낸 거절 요청이라, 아래에서 기다린 뒤에도 60초 창 안에 남습니다)
time.sleep(5)
rejected_statuses = []
for _ in range(3):
    rejected = httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": FREE_KEY}, timeout=REQUEST_TIMEOUT)
    rejected_statuses.append(rejected.status_code)
assert rejected_statuses == [429, 429, 429], f"한도에 닿은 free 키를 세 번 더 보낸 상태 코드가 {rejected_statuses}입니다. 60초 창 안에 통과한 요청이 3개면 429 여야 합니다(TODO (2))."

# 마지막 429 의 Retry-After 초만큼 기다린 뒤 다시 보냅니다. 가장 오래된 통과 요청이 창을 벗어나 창 안의 통과 요청이 2개(한도 3 미만)이므로 200 이어야 합니다
time.sleep(int(rejected.headers["retry-after"]))
after_wait = httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": FREE_KEY}, timeout=REQUEST_TIMEOUT)
assert after_wait.status_code == 200, f"Retry-After({rejected.headers['retry-after']}초)만큼 기다린 뒤 보낸 free 키 요청이 {after_wait.status_code}입니다. 가장 오래된 통과 요청이 창을 벗어나 창 안의 통과 요청이 2개뿐이므로 받아야 합니다. 429 로 거절한 요청까지 request_log 에 기록하고 있지 않은지 확인하세요(통과한 요청만 기록, TODO (2))."

# pro 키: 양식이 틀린 요청 네 번(vLLM 에 보내지 않습니다). 키와 한도를 통과한 뒤 본문 검사에서 422 가 됩니다
pro_statuses = []
for _ in range(4):
    response = httpx.post(CHAT_URL, json={"question": "환"}, headers={"X-API-Key": PRO_KEY}, timeout=REQUEST_TIMEOUT)
    pro_statuses.append(response.status_code)
assert pro_statuses == [422, 422, 422, 422], f"pro 키 요청 네 번의 상태 코드가 {pro_statuses}입니다. 500 이면 TODO (1), (2)를, 429 면 한도를 요금제의 requests_per_minute 에서 읽는지 확인하세요(TODO (2))."
print("✅ 통과!")

## 3. 한 양식의 오류 응답

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 401, 404, 422, 429와 vLLM 오류(502, 504)를 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;error&quot;: {&quot;code&quot;: ..., &quot;message&quot;: ...}}</code> 한 양식으로 돌려주면, 쇼핑몰 화면이 오류마다 본문 모양을 따로 처리하지 않아도 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 프로그램이 읽는 짧은 오류 이름이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">message</code>는 사람이 읽는 안내 문장입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/core/errors.py</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">error_response(status_code, code, message, headers=None)</code></strong>: 본문이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;error&quot;: {&quot;code&quot;: code, &quot;message&quot;: message}}</code>이고 상태 코드와 헤더를 그대로 담은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">JSONResponse</code>를 돌려줍니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">handle_http_error(request, exc)</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPException</code>을 오류 양식으로 바꿉니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ERROR_CODES</code>(401 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;invalid_service_key&quot;</code>, 404 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;not_found&quot;</code>, 429 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;rate_limited&quot;</code>)에서 꺼내되 없으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;http_error&quot;</code>를 쓰고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">message</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">exc.detail</code>을 글자로 바꾼 값입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">exc.headers</code>를 그대로 넘겨 429의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code>가 남게 합니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">handle_validation_error(request, exc)</code>:</strong> 상태 코드 422, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;invalid_request&quot;</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">message</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">exc.errors()</code>의 첫 번째 문제에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">loc</code>의 마지막 값(항목 이름)과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">msg</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;항목 이름: 내용&quot;</code>으로 이은 글자입니다. 질문이 너무 짧으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;question: &quot;</code>으로 시작합니다.</li>
<li style="margin:3px 0;">(4) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">handle_vllm_timeout(request, exc)</code>:</strong> 상태 코드 504, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;backend_timeout&quot;</code>입니다.</li>
<li style="margin:3px 0;">(5) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">handle_vllm_unreachable(request, exc)</code>:</strong> 상태 코드 502, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;backend_unavailable&quot;</code>입니다.</li>
<li style="margin:3px 0;">(6) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">register_error_handlers(app)</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app.add_exception_handler</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StarletteHTTPException</code>에 (2), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RequestValidationError</code>에 (3), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">openai.APITimeoutError</code>에 (4), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">openai.APIConnectionError</code>에 (5), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">openai.APIStatusError</code>에 제공한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">handle_vllm_status_error</code>를 붙입니다.</li>
</ul>
<p style="margin:6px 0;">(4), (5)는 4번에서 서버를 다른 설정으로 다시 띄워 확인하므로, 이 문항의 자가채점은 401, 404, 422, 429만 봅니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 02의 4절(오류 응답 양식)과 5절(vLLM이 응답하지 않을 때)을 참고합니다.
세부구현
1) error_response는 JSONResponse의 status_code, content, headers 키워드 인자를 씁니다.
2) 오류 이름은 딕셔너리의 get 메서드에 기본값을 주어 꺼냅니다.
3) 양식 오류의 첫 번째 문제는 exc.errors()의 0번 딕셔너리이고, 항목 이름은 그 loc의 마지막 값입니다.
4) 처리 함수마다 error_response를 불러 돌려줍니다.
5) add_exception_handler에는 예외 클래스와 처리 함수를 차례로 넘깁니다. APITimeoutError는 APIConnectionError의 한 종류이므로 둘 다 붙여야 시간 초과만 504가 됩니다.
```

</details>

In [ ]:
# [자가채점] 401, 404, 422, 429 응답 본문이 오류 양식인지와 Retry-After 가 남는지 확인합니다
cases = [
    ("키 없는 요청", httpx.post(CHAT_URL, json={"question": sample_question}, timeout=REQUEST_TIMEOUT), 401, "invalid_service_key"),
    ("없는 주소", httpx.get(f"{SERVING_URL}/v1/shop/unknown", timeout=REQUEST_TIMEOUT), 404, "not_found"),
    ("짧은 질문", httpx.post(CHAT_URL, json={"question": "환"}, headers={"X-API-Key": PRO_KEY}, timeout=REQUEST_TIMEOUT), 422, "invalid_request"),
]
for name, response, status, code in cases:
    assert response.status_code == status, f"{name}이 {response.status_code}입니다. {status} 여야 합니다. 404 면 secure_main.py 의 TODO (2)에서 shop_router 를 붙였는지, 500 이면 터미널 2의 마지막 오류 줄에서 비어 있는 TODO 를 확인하세요."
    body = response.json()
    assert list(body) == ["error"], f"{name}의 본문이 {body}입니다. {{\"detail\": ...}} 이면 register_error_handlers(TODO (6))를, 키가 다르면 error_response(TODO (1))를 확인하세요."
    assert body["error"]["code"] == code, f"{name}의 code 가 {body['error']['code']!r}입니다. {code!r} 여야 합니다(TODO (2), (3))."
    assert isinstance(body["error"]["message"], str) and len(body["error"]["message"]) > 0, f"{name}의 message 가 비어 있습니다."

# (3) 양식 오류의 message 는 "항목 이름: 내용"
short_message = cases[2][1].json()["error"]["message"]
assert short_message.startswith("question: "), f"짧은 질문의 message 가 {short_message!r}입니다. loc 의 마지막 값과 msg 를 \"항목 이름: 내용\" 으로 이으세요(TODO (3))."

# (2) 429: free 키로 양식이 틀린 요청을 429 가 나올 때까지 보냅니다(vLLM 에 보내지 않지만 한도는 씁니다)
free_statuses = []
limited = None
for _ in range(4):
    response = httpx.post(CHAT_URL, json={"question": "환"}, headers={"X-API-Key": FREE_KEY}, timeout=REQUEST_TIMEOUT)
    free_statuses.append(response.status_code)
    if response.status_code == 429:
        limited = response
        break
assert limited is not None, f"free 키 요청의 상태 코드가 {free_statuses}입니다. 네 번 안에 429 가 와야 합니다. 2번 자가채점부터 다시 통과시키세요."
assert limited.json() == {"error": {"code": "rate_limited", "message": limited.json()["error"]["message"]}}, f"429 본문이 {limited.json()}입니다. code 는 \"rate_limited\" 여야 합니다(TODO (2))."
assert "retry-after" in limited.headers, "오류 양식으로 바꾼 429 응답에 Retry-After 헤더가 없습니다. handle_http_error 가 exc.headers 를 error_response 에 넘기는지 확인하세요(TODO (2))."
print("✅ 통과!")

## 4. 환경 변수만 바꿔 vLLM 오류 확인하기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 배포하면 환경마다 vLLM 주소와 시간 제한이 다르므로, 코드는 그대로 두고 환경 변수만 바꿔 같은 서버를 실행합니다. vLLM에 닿지 않으면 502, 제한 시간 안에 답이 없으면 504가 3번의 오류 양식으로 돌아오는지 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;">틀린 vLLM 주소로 실행:</strong> 터미널 2의 서버를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고 아래 명령을 실행합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;환경 변수&gt;</code> 칸에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code>가 읽는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>을 아무도 듣지 않는 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8999/v1</code>로 주는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름=값</code>을 씁니다. 상담 주소 두 개는 502와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;backend_unavailable&quot;</code>을 돌려주고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>는 vLLM을 부르지 않으므로 200입니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;">시간 제한 1초로 실행:</strong> 같은 방법으로 서버를 다시 띄우되 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;환경 변수&gt;</code> 칸에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1</code>로 주는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름=값</code>을 씁니다. 생각을 켠(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think=True</code>) 답 전체 요청은 1초 안에 끝나지 않으므로 504와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;backend_timeout&quot;</code>을 돌려줍니다.</li>
</ul>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 02의 5절(vLLM이 응답하지 않을 때)을 참고합니다.
세부구현
1) 셸에서는 명령 앞에 이름=값을 붙이면 그 명령에만 환경 변수가 적용됩니다.
2) 변수 이름은 core/config.py의 Settings 필드 이름을 대문자로 쓴 것과 같아야 합니다.
```

</details>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2의 서버를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;환경 변수&gt;</code> 칸에 (1)의 값을 써서 다시 띄웁니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">고쳐서 실행 · RunPod 터미널 2</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">환경 변수 칸을 이름=값으로 바꾼 뒤 실행합니다</span></div>

```bash
<환경 변수> uv run uvicorn shop_api.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버를 다시 띄우면 요청 기록이 비워지므로 아래 자가채점은 pro 키 한도에 걸리지 않습니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# [자가채점] (1) vLLM 에 닿지 않을 때: 상담 주소 두 개는 502 backend_unavailable, 상태 확인은 200
unreachable = httpx.post(CHAT_URL, json={"question": sample_question}, headers={"X-API-Key": PRO_KEY}, timeout=REQUEST_TIMEOUT)
assert unreachable.status_code == 502, f"답 전체 요청이 {unreachable.status_code}입니다. 404 면 secure_main.py 의 TODO (2)에서 shop_router 를 붙였는지, 200 이면 VLLM_BASE_URL 을 명령 앞에 붙여 서버를 다시 띄웠는지, 500 이면 core/errors.py 의 TODO (5), (6)을 확인하세요."
assert unreachable.json()["error"]["code"] == "backend_unavailable", f"502 본문이 {unreachable.json()}입니다. code 는 \"backend_unavailable\" 여야 합니다(core/errors.py 의 TODO (5))."

# 스트리밍 주소도 스트림을 응답보다 먼저 열므로, 조각을 보내기 전에 502 로 알립니다
stream_unreachable = httpx.post(STREAM_URL, json={"question": sample_question}, headers={"X-API-Key": PRO_KEY}, timeout=REQUEST_TIMEOUT)
assert stream_unreachable.status_code == 502, f"스트리밍 요청이 {stream_unreachable.status_code}입니다. 200 이면 VLLM_BASE_URL 을 붙여 다시 띄웠는지, 500 이면 core/errors.py 의 TODO (5), (6)을 확인하세요. routers/shop_router.py 는 고치지 않습니다."
assert stream_unreachable.json()["error"]["code"] == "backend_unavailable", f"스트리밍 502 본문이 {stream_unreachable.json()}입니다."

# 상태 확인 주소는 vLLM 을 부르지 않으므로 그대로 200
health = httpx.get(f"{SERVING_URL}/healthz", timeout=REQUEST_TIMEOUT)
assert health.status_code == 200, f"GET /healthz 가 {health.status_code}입니다. 상태 확인 주소는 vLLM 이 없어도 200 이어야 합니다."
print("✅ 통과!")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2의 서버를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;환경 변수&gt;</code> 칸에 (2)의 값을 써서 다시 띄웁니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">고쳐서 실행 · RunPod 터미널 2</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">환경 변수 칸을 이름=값으로 바꾼 뒤 실행합니다</span></div>

```bash
<환경 변수> uv run uvicorn shop_api.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# [자가채점] (2) vLLM 이 제한 시간(1초) 안에 답하지 않을 때: 생각을 켠 답 전체 요청은 504 backend_timeout
slow = httpx.post(CHAT_URL, json={"question": sample_question, "think": True}, headers={"X-API-Key": PRO_KEY}, timeout=REQUEST_TIMEOUT)
assert slow.status_code == 504, f"생각을 켠 요청이 {slow.status_code}입니다. 404 면 secure_main.py 의 TODO (2)에서 shop_router 를 붙였는지, 200 이면 VLLM_TIMEOUT_SECONDS=1 을 명령 앞에 붙여 서버를 다시 띄웠는지, 502 면 core/errors.py 의 TODO (6)에서 APITimeoutError 를 붙였는지 확인하세요."
assert slow.json()["error"]["code"] == "backend_timeout", f"504 본문이 {slow.json()}입니다. code 는 \"backend_timeout\" 이어야 합니다(core/errors.py 의 TODO (4))."
print("✅ 통과!")

## 마무리

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2의 서버를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고 환경 변수 없이 다시 띄우면, 원래 vLLM 주소와 시간 제한(120초)으로 돌아갑니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn shop_api.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">오늘 실습을 여기서 마칠 때</summary>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>(FastAPI 서버)와 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> vLLM 서버)에서 차례로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> vLLM 서버가 꺼진 것입니다.</p>
<p style="margin:6px 0;">오늘 실습을 마치면 RunPod 콘솔에서 파드를 종료(Terminate)합니다. 정지(Stop)해도 다음에 같은 GPU로 다시 켜지지 않는 경우가 많습니다. 남길 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output</code> 등)이 있으면 종료 전에 내 PC로 받아 둡니다.</p>
</div>

</details>